# Portfolio analytics

This notebook only orchestrates: it calls `transactions` (trade logic), `prices` (Yahoo Finance fetch + cache), `returns` (CAGR / HYSA benchmark math), and `visualization` (all charts). No logic lives in this notebook itself — see `docs/architecture.md` for the module map.

In [1]:
from datetime import date
import polars as pl

from trades import returns, transactions, visualization
from trades.brokers.ibkr import main
from trades.config import AppConfig
from trades.market_data import prices
from trades.utils.frames import collect_if_lazy

AS_OF_DATE = date.today()  # change this to price the portfolio as of any past date

# Every tunable parameter lives on one of these config objects (see
# docs/architecture.md#configuration) — nothing here is a hidden default.
config = AppConfig()

## 1. Load, enrich, and aggregate trades

Trades come from the local IBKR ledger cache (`data/brokers/ibkr/ledger.csv`, kept fresh by running `notebooks/ibkr_sync.ipynb`), standardized from the canonical ledger onto the older, narrower trade schema by `preprocessing.standardize_ibkr_trades` (only `BUY` events on a real symbol — see `docs/architecture.md`, "Canonical trade schema"). `enrich_trades` then adds `usd_per_share`, and `aggregate_same_day_trades` merges same-day, same-symbol fills executed within 0.01% of each other into one row (summed shares/USD, recomputed $/share) — this is the dataset used for everything downstream.

In [2]:
ledger = main.load_ledger(config)
raw = transactions.standardize_ibkr_trades(ledger, config)
enriched = transactions.enrich_trades(raw)
trades = collect_if_lazy(transactions.aggregate_same_day_trades(enriched, config))
print(f"{len(collect_if_lazy(raw))} standardized buys -> {len(trades)} aggregated trades")
trades

25 standardized buys -> 15 aggregated trades


/Users/gabrielduguey/Documents/Perso/IBKR/src/trades/brokers/ibkr/main.py:70: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("meta").map_elements(json.loads)
with this one instead:
  + pl.col("meta").str.json_decode()

  return ledger.with_columns(pl.col("meta").map_elements(json.loads, return_dtype=pl.Object))


trade_date,symbol,shares,usd_spent,n_trades,usd_per_share
date,str,f64,f64,u32,f64
2026-01-27,"""VOO""",0.15,96.0585,1,640.39
2026-04-10,"""BND""",1.695,124.997775,1,73.745
2026-04-10,"""VOO""",1.3974,874.968036,1,626.14
2026-04-10,"""VXUS""",3.0706,249.992899,1,81.415
2026-05-05,"""BND""",3.4129,249.994925,2,73.25
…,…,…,…,…,…
2026-06-04,"""BND""",0.012,0.87828,1,73.19
2026-06-24,"""VXUS""",0.029,2.448615,1,84.435
2026-06-30,"""VOO""",4.9333,3363.573273,3,681.81


## 2. Investment schedule

Totals per symbol, invested-per-month (overall and per symbol), the daily investment timeline (with gaps between buys), and a pie breakdown with a menu to switch between whole-portfolio-by-symbol and any one symbol's by-date split.

In [3]:
total_by_symbol = transactions.total_invested_by_symbol(trades)
print("Total invested to date, by symbol:")
print(total_by_symbol)
print(f"\nTotal invested to date, whole portfolio: ${total_by_symbol.sum():,.2f}")

Total invested to date, by symbol:
shape: (4,)
Series: 'total_invested' [f64]
[
	12716.191108
	1889.9944
	1652.433236
	376.15763
]

Total invested to date, whole portfolio: $16,634.78


In [4]:
monthly = transactions.monthly_invested(trades)
monthly

month,VOO,BND,VXUS,QQQM,Total
str,f64,f64,f64,f64,f64
"""2026-01""",96.0585,0.0,0.0,0.0,96.0585
"""2026-04""",874.968036,124.997775,249.992899,0.0,1249.95871
"""2026-05""",1749.967847,250.281575,499.991804,0.0,2500.241226
"""2026-06""",9976.507145,0.87828,902.448533,1889.9944,12769.828358
"""2026-07""",18.68958,0.0,0.0,0.0,18.68958


In [5]:
visualization.plot_monthly_invested(monthly).show()

In [6]:
daily = transactions.daily_investment_timeline(trades)
visualization.plot_daily_investment_timeline(daily).show()

In [7]:
pie_options = transactions.pie_chart_options(trades)
visualization.plot_investment_pie(pie_options).show()

## 3. Price history

One local cache file per symbol (`data/prices/{SYMBOL}.csv`), each call only fetching the date range missing since the last run — see `docs/architecture.md` for the cache design. History goes back to the first trade date across the whole portfolio.

In [8]:
symbols = trades["symbol"].unique().sort().to_list()
first_trade_date = trades["trade_date"].min()

price_histories = prices.update_price_caches(
    symbols, since=first_trade_date, as_of=AS_OF_DATE, config=config
)
for symbol, history in price_histories.items():
    latest_close = history["close"][-1]
    print(f"{symbol}: {len(history)} trading days cached, latest close {latest_close:.2f}")

BND: 109 trading days cached, latest close 73.11
QQQM: 109 trading days cached, latest close 293.42
VOO: 109 trading days cached, latest close 684.84
VXUS: 109 trading days cached, latest close 84.84


## 4. Returns vs. a HYSA benchmark

For each trade: current price, days held, total return, CAGR-style annualized return, the compounded HYSA return over the same window (rate set by `returns_config.hysa_annual_rate`, default 4%), and the resulting alpha. See `docs/returns.md` for the derivation of each step.

In [9]:
def price_lookup(symbol: str, as_of: date) -> float | None:
    return prices.price_as_of(price_histories[symbol], as_of)


returns_df = returns.build_returns_table(
    trades, price_lookup, as_of=AS_OF_DATE, config=config
)

display_table = (
    returns_df
    .select([
        "trade_date",
        "symbol",
        "usd_per_share",
        "current_price",
        "days_held",
        "total_return_pct",
        "annualized_return_pct",
        "hysa_period_return_pct",
        "alpha_period_pct",
    ])
    .rename({"usd_per_share": "price_paid"})
)
display_table

trade_date,symbol,price_paid,current_price,days_held,total_return_pct,annualized_return_pct,hysa_period_return_pct,alpha_period_pct
date,str,f64,f64,i64,f64,f64,f64,f64
2026-01-27,"""VOO""",640.39,684.840027,158,6.941087,16.769038,1.712267,5.22882
2026-04-10,"""BND""",73.745,73.110001,85,-0.861074,-3.645459,0.917543,-1.778617
2026-04-10,"""VOO""",626.14,684.840027,85,9.374904,46.932249,0.917543,8.457362
2026-04-10,"""VXUS""",81.415,84.839996,85,4.206837,19.357152,0.917543,3.289294
2026-05-05,"""BND""",73.25,73.110001,60,-0.191125,-1.157046,0.646807,-0.837932
…,…,…,…,…,…,…,…,…
2026-06-04,"""BND""",73.19,73.110001,30,-0.109304,-1.321776,0.322882,-0.432186
2026-06-24,"""VXUS""",84.435,84.839996,10,0.479655,19.083576,0.107512,0.372143
2026-06-30,"""VOO""",681.81,684.840027,4,0.444409,49.874041,0.042991,0.401418


In [10]:

numeric_cols = [
    col
    for col, dtype in display_table.schema.items()
    if dtype in (pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                 pl.Float32, pl.Float64)
]

display_rounded = display_table.with_columns([
    pl.col(c).round(2).alias(c)
    for c in numeric_cols
])

visualization.render_table(display_rounded, title=f"Returns as of {AS_OF_DATE}").show() # type: ignore

portfolio_alpha = returns.portfolio_alpha_pct(returns_df)
label = (
    f"Dollar-weighted portfolio alpha vs. {config.returns.hysa_annual_rate:.0%} HYSA "
    "(period, not annualized)"
)
print(f"{label}: {portfolio_alpha:+.2f}%")

Dollar-weighted portfolio alpha vs. 4% HYSA (period, not annualized): -0.62%


## 5. Annualized return curve

Per-trade annualized return against days held, with a fitted trend and the flat HYSA benchmark line. Short holds annualize into large, noisy numbers by design — that's why the combined alpha above uses period alpha instead of this annualized figure.

In [11]:
trend_x, trend_y = returns.fit_trend(
    returns_df.select("days_held").to_series().to_numpy(),
    returns_df.select("annualized_return_pct").to_series().to_numpy(),
    config,
)
visualization.plot_return_curve(returns_df, trend_x, trend_y, config).show()